# GeoGuessr v7 - fixes the three calibration bugs from the 78-point run

What changed and why:

1. **True holdout.** The old calibration fitted `alpha` on single-model out-of-fold
   predictions, then applied it to a 5-model average at test time. Averaging
   compresses the uncertainty head's spread, so the emitted radii never matched
   the distribution alpha was fitted to (OOF median radius 905 km vs 638 km on
   test). Now a block of provided images is held out from **every** model, and the
   policy is fitted on the *ensemble's* predictions over it - the exact object
   used at test time.
2. **Joint policy search.** lambda used to be picked against a placeholder radius
   of 2.0x, then alpha settled at 4.2. Now (blend, lambda, alpha, floor) are
   searched together.
3. **Fine-tune is blended, not discarded.** The previous run's fine-tuned model
   scored 0.2784 against stage 1's 0.3064 - close, and it was cut off mid-LR
   schedule by the time guard, so it was undertrained rather than bad. Averaging
   two models that fail differently beats picking one.

Plus: DINOv2 as a second backbone, a sweep over how much to up-weight the
in-domain provided images, and test-time augmentation that is only enabled if it
helps on the holdout.

**Attach your previous run's output as a dataset** (Add Input -> Your Work ->
the notebook that scored 78). This reuses `feat_train.npy` and skips the
117-minute extraction, and reuses `finetuned.pt` as an ensemble member.
Everything degrades gracefully if those files are absent.

In [1]:
# =====================================================================
# CELL 1 - Installs
# =====================================================================
!pip install -q transformers==4.44.2 huggingface_hub shapely scikit-learn pandas numpy pillow tqdm openpyxl scipy
print("installs done", flush=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 604.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 104.3 MB/s eta 0:00:00
installs done


In [2]:
# =====================================================================
# CELL 2 - Config, logging, geometry
# =====================================================================
import os, sys, gc, io, json, math, time, glob, random, zipfile, warnings, traceback
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
from tqdm.auto import tqdm
from sklearn.cluster import MiniBatchKMeans

T0 = time.time()
def log(m):
    print(f"[{time.strftime('%H:%M:%S')} | +{(time.time()-T0)/60:6.1f} min] {m}", flush=True)

class CFG:
    SEED = 42
    WORK = "/kaggle/working"; ART = "/kaggle/working/artifacts"; TMP = "/kaggle/temp"

    CLIP_ID  = "openai/clip-vit-large-patch14"
    DINO_ID  = "facebook/dinov2-large"
    USE_DINO = True            # second backbone: decorrelated errors
    IMG = 224; EXTRACT_BS = 48; WORKERS = 4

    USE_EXTERNAL = True
    EXT_REPO = "osv5m/osv5m"; EXT_SHARDS = 5; MAX_EXTERNAL = 250_000

    N_FINE = 2000; N_COARSE = 150; TAU_KM = 250.0; TOP_K = 8

    N_MODELS = 5               # seed ensemble, all trained on all non-holdout data
    EPOCHS = 60; HEAD_BS = 2048; LR = 2e-3; WD = 1e-4; HID = 1024
    W_PROVIDED_GRID = [3.0, 8.0, 20.0]
    W_EXTERNAL = 1.0
    Q_UNC = 0.70
    R_MAX = 3000.0

    USE_FT_MEMBER = True       # blend in the previous run's fine-tuned model
    RUN_PHASH_GUARD = True

R_EARTH = 6371.0088
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
seed_all(CFG.SEED)
os.makedirs(CFG.ART, exist_ok=True); os.makedirs(CFG.TMP, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log(f"device={device} | {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")

def latlon_to_vec(lat, lon):
    la = np.radians(np.asarray(lat, float)); lo = np.radians(np.asarray(lon, float))
    return np.stack([np.cos(la)*np.cos(lo), np.cos(la)*np.sin(lo), np.sin(la)], -1)
def vec_to_latlon(v):
    v = np.asarray(v, float); v = v/(np.linalg.norm(v, axis=-1, keepdims=True)+1e-12)
    return (np.degrees(np.arcsin(np.clip(v[...,2],-1,1))),
            np.degrees(np.arctan2(v[...,1], v[...,0])))
def hav_km(a,b,c,d):
    a,b,c,d = map(lambda x: np.radians(np.asarray(x,float)), (a,b,c,d))
    h = np.sin((c-a)/2)**2 + np.cos(a)*np.cos(c)*np.sin((d-b)/2)**2
    return 2*R_EARTH*np.arcsin(np.sqrt(np.clip(h,0,1)))
log("config ready")

[02:32:00 | +   0.0 min] device=cuda | Tesla T4
[02:32:00 | +   0.0 min] config ready


In [3]:
# =====================================================================
# CELL 3 - Discover competition files + previous-run artifacts
# =====================================================================
def find_all(pat, root="/kaggle/input"):
    return sorted(glob.glob(os.path.join(root,"**",pat), recursive=True))
def pick_col(cols, *k):
    for c in cols:
        lc = c.lower().replace("_","").replace(" ","")
        if all(x in lc for x in k): return c

SAMPLE_SUB = find_all("sample*submission*.csv")[0]
GEOJSON    = (find_all("*.geojson") or [None])[0]
sub_template = pd.read_csv(SAMPLE_SUB)
SUB_ID  = pick_col(sub_template.columns,"id") or sub_template.columns[0]
SUB_LAT = pick_col(sub_template.columns,"lat")
SUB_LON = pick_col(sub_template.columns,"lon") or pick_col(sub_template.columns,"lng")
SUB_RAD = pick_col(sub_template.columns,"rad")
log(f"submission cols: {list(sub_template.columns)} -> {SUB_ID}/{SUB_LAT}/{SUB_LON}/{SUB_RAD}")

GT_PATH = [p for p in find_all("*.csv")+find_all("*.xlsx")
           if "ground" in os.path.basename(p).lower()
           or "coordinate" in os.path.basename(p).lower()][0]
gt = pd.read_excel(GT_PATH) if GT_PATH.endswith(".xlsx") else pd.read_csv(GT_PATH)
GT_LAT = pick_col(gt.columns,"lat"); GT_LON = pick_col(gt.columns,"lon") or pick_col(gt.columns,"lng")
GT_ID  = pick_col(gt.columns,"id") or gt.columns[0]

# ---- previous run's artifacts (optional but saves ~2 hours) ----
PREV_FEAT = [p for p in find_all("feat_train.npy") if "/kaggle/input/" in p]
PREV_FT   = [p for p in find_all("finetuned.pt")   if "/kaggle/input/" in p]
PREV_CLIP = [p for p in find_all("clip_vit_l14")   if "/kaggle/input/" in p]
log(f"reusable: feat_train={bool(PREV_FEAT)} finetuned={bool(PREV_FT)} clip={bool(PREV_CLIP)}")

ALL_IMGS = find_all("*.jpg")+find_all("*.jpeg")+find_all("*.png")
by_stem = {}
for p in ALL_IMGS:
    by_stem.setdefault(os.path.splitext(os.path.basename(p))[0], p)
test_ids = sub_template[SUB_ID].astype(str).tolist()
test_stem_set = set(os.path.splitext(t)[0] for t in test_ids)
TEST_PATHS = [by_stem.get(os.path.splitext(str(t))[0]) for t in test_ids]
log(f"test images matched: {sum(p is not None for p in TEST_PATHS)}/{len(TEST_PATHS)}")

gt["stem"] = gt[GT_ID].astype(str).map(lambda x: os.path.splitext(str(x))[0])
gt["path"] = gt["stem"].map(by_stem.get)
gt = gt[gt["path"].notna() & ~gt["stem"].isin(test_stem_set)].copy()
train_df = pd.DataFrame({"path": gt["path"].values,
                         "lat": pd.to_numeric(gt[GT_LAT], errors="coerce").values,
                         "lon": pd.to_numeric(gt[GT_LON], errors="coerce").values,
                         "src": "provided"}).dropna(subset=["lat","lon"])
train_df = train_df[train_df.lat.between(-90,90) & train_df.lon.between(-180,180)].reset_index(drop=True)
log(f"provided training rows: {len(train_df)}")

[02:32:18 | +   0.3 min] submission cols: ['image_id', 'pred_lat', 'pred_lon', 'pred_radius_km'] -> image_id/pred_lat/pred_lon/pred_radius_km
[02:32:25 | +   0.4 min] reusable: feat_train=True finetuned=True clip=True
[02:32:25 | +   0.4 min] test images matched: 500/500
[02:32:25 | +   0.4 min] provided training rows: 19002


In [4]:
# =====================================================================
# CELL 4 - Backbones, saved locally (rule 4.3)
# =====================================================================
from transformers import CLIPVisionModel, AutoModel
CLIP_LOCAL = os.path.join(CFG.ART, "clip_vit_l14")
DINO_LOCAL = os.path.join(CFG.ART, "dinov2_large")

if PREV_CLIP and not os.path.exists(os.path.join(CLIP_LOCAL,"config.json")):
    import shutil; shutil.copytree(PREV_CLIP[0], CLIP_LOCAL); log("CLIP copied from previous run")
if not os.path.exists(os.path.join(CLIP_LOCAL,"config.json")):
    CLIPVisionModel.from_pretrained(CFG.CLIP_ID).save_pretrained(CLIP_LOCAL)
log(f"CLIP local -> {CLIP_LOCAL}")
if CFG.USE_DINO and not os.path.exists(os.path.join(DINO_LOCAL,"config.json")):
    try: AutoModel.from_pretrained(CFG.DINO_ID).save_pretrained(DINO_LOCAL)
    except Exception as e: log(f"DINOv2 unavailable ({e})"); CFG.USE_DINO = False
gc.collect()

def load_backbone(kind):
    if kind == "clip":
        m = CLIPVisionModel.from_pretrained(CLIP_LOCAL).to(device).half().eval()
        mean, std = [0.48145466,0.4578275,0.40821073], [0.26862954,0.26130258,0.27577711]
    else:
        m = AutoModel.from_pretrained(DINO_LOCAL).to(device).half().eval()
        mean, std = [0.485,0.456,0.406], [0.229,0.224,0.225]
    for p in m.parameters(): p.requires_grad = False
    return dict(model=m, kind=kind, dim=m.config.hidden_size,
                mean=torch.tensor(mean, device=device).view(1,3,1,1).half(),
                std =torch.tensor(std,  device=device).view(1,3,1,1).half())
N_VIEWS = 2
log(f"backbones ready | dino={CFG.USE_DINO}")

[02:32:36 | +   0.6 min] CLIP copied from previous run
[02:32:36 | +   0.6 min] CLIP local -> /kaggle/working/artifacts/clip_vit_l14


config.json:   0%|          | 0.00/549 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

[02:32:57 | +   0.9 min] backbones ready | dino=True


In [5]:
# =====================================================================
# CELL 5 - External data
# =====================================================================
ext_df = pd.DataFrame(columns=["path","lat","lon","src"])
if CFG.USE_EXTERNAL:
    try:
        from huggingface_hub import list_repo_files, hf_hub_download
        files = list_repo_files(CFG.EXT_REPO, repo_type="dataset")
        zips  = sorted([f for f in files if f.endswith(".zip") and "train" in f.lower()])
        metas = sorted([f for f in files if f.endswith(".csv") and "train" in f.lower()])
        mp = hf_hub_download(CFG.EXT_REPO, metas[0], repo_type="dataset", local_dir=CFG.TMP)
        h  = pd.read_csv(mp, nrows=5)
        m_id = pick_col(h.columns,"id") or h.columns[0]
        meta = pd.read_csv(mp, usecols=[m_id, pick_col(h.columns,"lat"), pick_col(h.columns,"lon")])
        meta.columns = ["ext_id","lat","lon"]; meta["ext_id"] = meta["ext_id"].astype(str)
        ext_dir = os.path.join(CFG.TMP,"osv5m_imgs"); os.makedirs(ext_dir, exist_ok=True)
        for z in zips[:CFG.EXT_SHARDS]:
            zp = hf_hub_download(CFG.EXT_REPO, z, repo_type="dataset", local_dir=CFG.TMP)
            with zipfile.ZipFile(zp) as zf: zf.extractall(ext_dir)
            os.remove(zp); log(f"extracted {z}")
        ep = sorted(glob.glob(os.path.join(ext_dir,"**","*.jpg"), recursive=True))
        emap = pd.DataFrame({"path":ep, "ext_id":[os.path.splitext(os.path.basename(p))[0] for p in ep]})
        emap = emap[~emap.ext_id.isin(test_stem_set)]
        ext_df = emap.merge(meta, on="ext_id", how="inner").dropna(subset=["lat","lon"])
        if len(ext_df) > CFG.MAX_EXTERNAL:
            ext_df = ext_df.sample(CFG.MAX_EXTERNAL, random_state=CFG.SEED)
        ext_df = ext_df[["path","lat","lon"]]; ext_df["src"] = "external"
        log(f"external rows: {len(ext_df)}")
    except Exception:
        log("external data failed - provided only"); traceback.print_exc()

full_df = pd.concat([train_df, ext_df], ignore_index=True)
log(f"TOTAL rows = {len(full_df)} (provided={int((full_df.src=='provided').sum())}, "
    f"external={int((full_df.src=='external').sum())})")

train.csv:   0%|          | 0.00/2.92G [00:00<?, ?B/s]

images/train/00.zip:   0%|          | 0.00/2.52G [00:00<?, ?B/s]

[02:33:53 | +   1.9 min] extracted images/train/00.zip


images/train/01.zip:   0%|          | 0.00/2.52G [00:00<?, ?B/s]

[02:34:28 | +   2.5 min] extracted images/train/01.zip


...

[02:34:57 | +   3.0 min] extracted images/train/02.zip


images/train/03.zip:   0%|          | 0.00/2.51G [00:00<?, ?B/s]

[02:35:30 | +   3.5 min] extracted images/train/03.zip


images/train/04.zip:   0%|          | 0.00/2.52G [00:00<?, ?B/s]

[02:35:57 | +   4.0 min] extracted images/train/04.zip
[02:36:02 | +   4.0 min] external rows: 250000
[02:36:02 | +   4.0 min] TOTAL rows = 269002 (provided=19002, external=250000)


In [6]:
# =====================================================================
# CELL 6 - Contamination guard (content hash, not filenames)
# Last run this found 266 genuine overlaps - it is not decorative.
# =====================================================================
HAM_THRESH = 8
if CFG.RUN_PHASH_GUARD:
    try:
        from scipy.fftpack import dct
        POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
        def _ph(img, hs=8, size=32):
            g = img.convert("L").resize((size,size), Image.BICUBIC)
            d = dct(dct(np.asarray(g,dtype=np.float64),axis=0,norm="ortho"),axis=1,norm="ortho")[:hs,:hs]
            f = d.flatten()[1:]
            return np.packbits((f > np.median(f)).astype(np.uint8))
        class HDS(Dataset):
            def __init__(s,p): s.p=[x if x else "" for x in p]
            def __len__(s): return len(s.p)
            def __getitem__(s,i):
                try: return torch.tensor(_ph(Image.open(s.p[i]))), 1
                except Exception: return torch.zeros(8,dtype=torch.uint8), 0
        def hash_all(p, tag):
            dl=DataLoader(HDS(p),batch_size=256,num_workers=CFG.WORKERS); H,O,n=[],[],0
            for h,o in dl:
                H.append(h.numpy()); O.append(o.numpy()); n+=len(h)
                if n % 50000 < 256: log(f"  [{tag}] {n}/{len(p)}")
            return np.concatenate(H), np.concatenate(O)
        def minham(A,B):
            out=np.full(len(A),64,dtype=np.int16)
            for i in range(0,len(A),4096):
                out[i:i+4096]=POP[A[i:i+4096][:,None,:]^B[None,:,:]].sum(-1).min(1)
            return out
        Ht,ok = hash_all([p for p in TEST_PATHS if p],"test"); Ht=Ht[ok==1]
        Hx,_  = hash_all(full_df.path.values,"pool")
        hit = minham(Hx,Ht) <= HAM_THRESH
        log(f"CONTAMINATED rows: {int(hit.sum())} / {len(full_df)}")
        if hit.sum():
            full_df = full_df[~hit].reset_index(drop=True)
            log(f"dropped. clean pool = {len(full_df)}")
    except Exception as e:
        log(f"phash guard failed ({e})"); traceback.print_exc()
log(f"VERIFIED pool: {len(full_df):,}")

[02:37:53 | +   5.9 min]   [pool] 50176/269002
[02:39:17 | +   7.3 min]   [pool] 100096/269002
[02:40:37 | +   8.6 min]   [pool] 150016/269002
[02:41:55 | +   9.9 min]   [pool] 200192/269002
[02:43:12 | +  11.2 min]   [pool] 250112/269002
[02:43:49 | +  11.8 min] CONTAMINATED rows: 266 / 269002
[02:43:49 | +  11.8 min] dropped. clean pool = 268736
[02:43:49 | +  11.8 min] VERIFIED pool: 268,736


In [7]:
# =====================================================================
# CELL 7 - Country labels
# =====================================================================
import shapely
from shapely.geometry import shape, Point
from shapely.strtree import STRtree
from shapely.ops import nearest_points

country_geoms = []
if GEOJSON:
    for ft_ in json.load(open(GEOJSON,encoding="utf-8"))["features"]:
        try:
            g = shape(ft_["geometry"]); country_geoms.append(g if g.is_valid else g.buffer(0))
        except Exception: pass
CTREE = STRtree(country_geoms) if country_geoms else None
N_COUNTRY = max(len(country_geoms), 1)
log(f"country polygons: {len(country_geoms)}")

def assign_country(lats, lons):
    out = np.full(len(lats), -1, dtype=np.int64)
    if CTREE is None: return out
    pts = shapely.points(np.asarray(lons,float), np.asarray(lats,float))
    try:
        pr = CTREE.query(pts, predicate="intersects"); out[pr[0]] = pr[1]
    except Exception:
        for i,p in enumerate(pts):
            h = CTREE.query(p, predicate="intersects")
            if len(h): out[i] = int(h[0])
    return out

full_df["country"] = assign_country(full_df.lat.values, full_df.lon.values)
log(f"inside a country: {int((full_df.country>=0).sum())}/{len(full_df)}")

[02:43:51 | +  11.8 min] country polygons: 298
[02:44:20 | +  12.3 min] inside a country: 261874/268736


In [8]:
# =====================================================================
# CELL 8 - Geocells + TRUE HOLDOUT
#
# The holdout is reproduced as the old run's fold 0, so the previous
# fine-tuned model has not seen it either - otherwise blending it in would
# be calibrated against data it trained on.
# =====================================================================
V = latlon_to_vec(full_df.lat.values, full_df.lon.values)
km_fine = MiniBatchKMeans(CFG.N_FINE, random_state=CFG.SEED, batch_size=8192,
                          n_init=5, max_iter=300).fit(V)
full_df["fine"] = km_fine.labels_
CF = km_fine.cluster_centers_/(np.linalg.norm(km_fine.cluster_centers_,axis=1,keepdims=True)+1e-12)
km_coarse = MiniBatchKMeans(CFG.N_COARSE, random_state=CFG.SEED, batch_size=8192,
                            n_init=5, max_iter=300).fit(V)
full_df["coarse"] = km_coarse.labels_

cosm = np.clip(CF @ CF.T, -1, 1)
S = np.exp(-(R_EARTH*np.arccos(cosm))/CFG.TAU_KM); S /= S.sum(1, keepdims=True)
SOFT = torch.tensor(S, dtype=torch.float32, device=device)
CFT  = torch.tensor(CF, dtype=torch.float32, device=device)

_cl, _co = vec_to_latlon(CF)
CELL_COUNTRY = assign_country(_cl, _co)
CCT = torch.tensor(np.where(CELL_COUNTRY>=0, CELL_COUNTRY, 0), device=device)
CC_VALID = torch.tensor((CELL_COUNTRY>=0).astype(np.float32), device=device)
log(f"cells on land: {(CELL_COUNTRY>=0).sum()}/{CFG.N_FINE}")

prov = (full_df.src.values == "provided")
n_blocks = min(2000, max(6, int(prov.sum()//12)))
kb = MiniBatchKMeans(n_blocks, random_state=CFG.SEED, batch_size=4096, n_init=5).fit(V[prov])
blocks = np.full(len(full_df), -1, dtype=np.int64); blocks[prov] = kb.labels_

# reproduce the previous run's fold-0 => our untouched holdout
perm = np.random.RandomState(CFG.SEED).permutation(n_blocks)
old_fold = {b: i % 5 for i, b in enumerate(perm)}
is_hold = np.zeros(len(full_df), bool)
is_hold[prov] = np.array([old_fold[b] == 0 for b in blocks[prov]])
full_df["holdout"] = is_hold
HOLD = np.where(is_hold)[0]
TRAINABLE = np.where(~is_hold)[0]
log(f"TRUE HOLDOUT: {len(HOLD)} provided images, excluded from every model")
log(f"trainable rows: {len(TRAINABLE)}")

[02:44:41 | +  12.7 min] cells on land: 1874/2000
[02:44:48 | +  12.8 min] TRUE HOLDOUT: 3908 provided images, excluded from every model
[02:44:48 | +  12.8 min] trainable rows: 264828


In [9]:
# =====================================================================
# CELL 9 - Features (reuse the cached CLIP matrix if it lines up)
# =====================================================================
class ViewDS(Dataset):
    """Full-frame squash + centre square. No flips: mirroring kills driving side."""
    def __init__(s, paths, jitter=0.0):
        s.p = [x if x else "" for x in paths]; s.j = jitter
    def __len__(s): return len(s.p)
    def __getitem__(s, i):
        try: im = Image.open(s.p[i]).convert("RGB")
        except Exception: im = Image.new("RGB",(CFG.IMG,CFG.IMG),(128,128,128))
        if s.j > 0:
            w,h = im.size; sc = 1.0 - s.j
            cw,ch = int(w*sc), int(h*sc)
            im = im.crop(((w-cw)//2, (h-ch)//2, (w-cw)//2+cw, (h-ch)//2+ch))
        w,h = im.size; q = min(w,h); l,t = (w-q)//2,(h-q)//2
        a = im.resize((CFG.IMG,CFG.IMG), Image.BICUBIC)
        b = im.crop((l,t,l+q,t+q)).resize((CFG.IMG,CFG.IMG), Image.BICUBIC)
        return torch.stack([torch.from_numpy(np.asarray(a,dtype=np.uint8)).permute(2,0,1),
                            torch.from_numpy(np.asarray(b,dtype=np.uint8)).permute(2,0,1)])

@torch.no_grad()
def encode_with(bk, paths, tag, jitter=0.0):
    dl = DataLoader(ViewDS(paths, jitter), batch_size=CFG.EXTRACT_BS, shuffle=False,
                    num_workers=CFG.WORKERS, pin_memory=True)
    out, seen, t0 = [], 0, time.time()
    for x in dl:
        B = x.shape[0]
        x = x.to(device, non_blocking=True).reshape(B*N_VIEWS,3,CFG.IMG,CFG.IMG).half().div_(255.)
        o = bk["model"](pixel_values=(x-bk["mean"])/bk["std"])
        f = o.pooler_output if bk["kind"]=="clip" else o.last_hidden_state[:,0]
        out.append(f.reshape(B, N_VIEWS*bk["dim"]).float().cpu().numpy().astype(np.float16))
        seen += B
        if seen % (CFG.EXTRACT_BS*150) < CFG.EXTRACT_BS:
            r = seen/max(time.time()-t0,1e-6)
            log(f"  [{tag}] {seen}/{len(paths)} | {r:.1f} img/s | eta {(len(paths)-seen)/max(r,1e-6)/60:.1f} min")
    return np.concatenate(out)

FEATS = []
# ---- CLIP ----
P = os.path.join(CFG.WORK,"feat_clip.npy")
reused = False
if PREV_FEAT:
    try:
        cand = np.load(PREV_FEAT[0], mmap_mode="r")
        if cand.shape[0] == len(full_df):
            FEATS.append(np.array(cand)); reused = True
            log(f"REUSED cached CLIP features {cand.shape} - saved ~2 hours")
        else:
            log(f"cached features have {cand.shape[0]} rows but pool has {len(full_df)} - re-extracting")
    except Exception as e:
        log(f"could not reuse cache ({e})")
if not reused:
    if os.path.exists(P): FEATS.append(np.load(P)); log("loaded local CLIP features")
    else:
        bk = load_backbone("clip")
        f = encode_with(bk, full_df.path.values, "clip"); np.save(P, f); FEATS.append(f)
        bk["model"].cpu(); del bk; gc.collect(); torch.cuda.empty_cache()

# ---- DINOv2 ----
if CFG.USE_DINO:
    PD = os.path.join(CFG.WORK,"feat_dino.npy")
    if os.path.exists(PD) and np.load(PD, mmap_mode="r").shape[0] == len(full_df):
        FEATS.append(np.load(PD)); log("loaded cached DINOv2 features")
    else:
        try:
            bk = load_backbone("dino")
            f = encode_with(bk, full_df.path.values, "dino"); np.save(PD, f); FEATS.append(f)
            bk["model"].cpu(); del bk; gc.collect(); torch.cuda.empty_cache()
        except Exception as e:
            log(f"DINOv2 extraction failed ({e}) - continuing with CLIP only"); CFG.USE_DINO = False

X = np.concatenate(FEATS,1) if len(FEATS)>1 else FEATS[0]
del FEATS; gc.collect()
MU = X.astype(np.float32).mean(0); SD = X.astype(np.float32).std(0)+1e-6
np.save(os.path.join(CFG.ART,"feat_mu.npy"),MU); np.save(os.path.join(CFG.ART,"feat_sd.npy"),SD)
Xg = torch.tensor((X.astype(np.float32)-MU)/SD, dtype=torch.float32, device=device)
FEAT_DIM = Xg.shape[1]
log(f"feature matrix on GPU: {tuple(Xg.shape)}")
del X; gc.collect()

[02:44:57 | +  13.0 min] REUSED cached CLIP features (268736, 2048) - saved ~2 hours
[02:48:27 | +  16.5 min]   [dino] 7200/268736 | 35.2 img/s | eta 123.8 min
[02:52:06 | +  20.1 min]   [dino] 14400/268736 | 34.0 img/s | eta 124.7 min
[02:55:45 | +  23.8 min]   [dino] 21600/268736 | 33.6 img/s | eta 122.5 min
[02:59:24 | +  27.4 min]   [dino] 28800/268736 | 33.4 img/s | eta 119.6 min
[03:03:02 | +  31.0 min]   [dino] 36000/268736 | 33.3 img/s | eta 116.3 min
[03:06:41 | +  34.7 min]   [dino] 43200/268736 | 33.3 img/s | eta 113.0 min
[03:10:20 | +  38.3 min]   [dino] 50400/268736 | 33.2 img/s | eta 109.6 min
[03:13:59 | +  42.0 min]   [dino] 57600/268736 | 33.2 img/s | eta 106.1 min
[03:17:39 | +  45.7 min]   [dino] 64800/268736 | 33.1 img/s | eta 102.6 min
[03:21:19 | +  49.3 min]   [dino] 72000/268736 | 33.1 img/s | eta 99.1 min
[03:24:58 | +  53.0 min]   [dino] 79200/268736 | 33.1 img/s | eta 95.6 min
[03:28:39 | +  56.7 min]   [dino] 86400/268736 | 33.0 img/s | eta 92.0 min
[03:32:

0

In [10]:
# =====================================================================
# CELL 10 - Heads, constrained decoding, training
# =====================================================================
class GeoHead(nn.Module):
    def __init__(s, d, nf, nc, nk, hid=1024):
        super().__init__()
        s.trunk = nn.Sequential(nn.Linear(d,hid), nn.LayerNorm(hid), nn.GELU(), nn.Dropout(0.30),
                                nn.Linear(hid,hid), nn.LayerNorm(hid), nn.GELU(), nn.Dropout(0.15))
        s.fine=nn.Linear(hid,nf); s.coarse=nn.Linear(hid,nc); s.country=nn.Linear(hid,nk)
        s.delta=nn.Linear(hid,3); s.unc=nn.Linear(hid,1)
    def forward(s,x):
        h=s.trunk(x)
        return s.fine(h), s.coarse(h), s.country(h), s.delta(h), s.unc(h).squeeze(-1)

def decode_point(fl, delta, topk=CFG.TOP_K, max_spread_km=600.0):
    p = torch.softmax(fl.float(),1); w, idx = torch.topk(p, topk, 1)
    anchor = CFT[idx[:,0]]; cand = CFT[idx]
    cos = (cand*anchor.unsqueeze(1)).sum(-1).clamp(-1+1e-9,1-1e-9)
    w = w*((R_EARTH*torch.acos(cos)) <= max_spread_km).float()
    w = w/(w.sum(1,keepdim=True)+1e-9)
    base = (cand*w.unsqueeze(-1)).sum(1); base = base/(base.norm(dim=1,keepdim=True)+1e-9)
    v = base + 0.08*torch.tanh(delta)
    return v/(v.norm(dim=1,keepdim=True)+1e-9)

def country_adjust(fl, cl, lam):
    if lam <= 0: return fl
    lp = torch.log_softmax(fl.float(),1); pc = torch.softmax(cl.float(),1)
    return lp + lam*torch.log(pc[:,CCT]*CC_VALID.unsqueeze(0) + 1e-6)

def km_between(a,b):
    return R_EARTH*torch.acos(torch.clamp((a*b).sum(1),-1+1e-9,1-1e-9))

y_fine = torch.tensor(full_df["fine"].values.astype(np.int64), device=device)
y_coar = torch.tensor(full_df["coarse"].values.astype(np.int64), device=device)
y_ctry = torch.tensor(full_df["country"].values.astype(np.int64), device=device)
y_vec  = torch.tensor(latlon_to_vec(full_df.lat.values, full_df.lon.values),
                      dtype=torch.float32, device=device)
src_prov = torch.tensor((full_df.src.values=="provided").astype(np.float32), device=device)

def train_model(idxs, seed, w_prov, epochs=CFG.EPOCHS, quiet=False):
    torch.manual_seed(seed); np.random.seed(seed)
    w_all = src_prov*w_prov + (1-src_prov)*CFG.W_EXTERNAL
    m = GeoHead(FEAT_DIM, CFG.N_FINE, CFG.N_COARSE, N_COUNTRY, CFG.HID).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=CFG.LR, weight_decay=CFG.WD)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG.LR,
            total_steps=epochs*max(1,len(idxs)//CFG.HEAD_BS), pct_start=0.25)
    for ep in range(epochs):
        m.train(); order = idxs[torch.randperm(len(idxs)).numpy()]; tot=nb=0
        for i in range(0, len(order)-CFG.HEAD_BS+1, CFG.HEAD_BS):
            b = torch.tensor(order[i:i+CFG.HEAD_BS], device=device)
            fl,cl,ctl,dl_,ul = m(Xg[b])
            l_f = -(SOFT[y_fine[b]]*torch.log_softmax(fl,1)).sum(1)
            l_c = F.cross_entropy(cl, y_coar[b], reduction="none")
            l_k = torch.where(y_ctry[b]>=0,
                    F.cross_entropy(ctl, y_ctry[b].clamp(min=0), reduction="none"),
                    torch.zeros_like(l_c))
            v = decode_point(fl, dl_); ch = (v-y_vec[b]).norm(dim=1)
            l_p = F.huber_loss(ch, torch.zeros_like(ch), reduction="none", delta=0.05)
            with torch.no_grad(): e = torch.log1p(km_between(v.detach(), y_vec[b]))
            d = e-ul; l_u = torch.maximum(CFG.Q_UNC*d, (CFG.Q_UNC-1)*d)
            loss = (w_all[b]*(l_f + 0.2*l_c + 1.5*l_k + 8.0*l_p + 0.2*l_u)).mean()
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 2.0); opt.step(); sch.step()
            tot += loss.item(); nb += 1
        if not quiet and (ep % 15 == 0 or ep == epochs-1):
            log(f"    seed{seed} ep {ep+1}/{epochs} | loss {tot/max(nb,1):.4f}")
    m.eval(); return m

@torch.no_grad()
def head_predict(m, idxs):
    out = [[],[],[],[]]
    for i in range(0, len(idxs), 8192):
        b = torch.tensor(idxs[i:i+8192], device=device)
        fl,cl,ctl,dl_,ul = m(Xg[b])
        out[0].append(torch.softmax(fl.float(),1)); out[1].append(torch.softmax(ctl.float(),1))
        out[2].append(dl_.float()); out[3].append(ul.float())
    return [torch.cat(o) for o in out]
log("training utilities ready")

[05:02:08 | + 150.1 min] training utilities ready


In [11]:
# =====================================================================
# CELL 11 - How much should in-domain images be up-weighted?
# One cheap model per candidate, judged on the true holdout.
# =====================================================================
hold_lat = full_df.lat.values[HOLD]; hold_lon = full_df.lon.values[HOLD]
hold_ctry = full_df.country.values[HOLD]

best_w, best_med = None, None
for w in CFG.W_PROVIDED_GRID:
    m = train_model(TRAINABLE, 1000, w, epochs=25, quiet=True)
    pf, pc, pd_, pu = head_predict(m, HOLD)
    v = decode_point(torch.log(pf+1e-12), pd_)
    la, lo = vec_to_latlon(v.cpu().numpy())
    med = float(np.median(hav_km(la, lo, hold_lat, hold_lon)))
    ok  = float(((assign_country(la,lo)==hold_ctry)&(hold_ctry>=0)).mean())
    log(f"  W_PROVIDED={w:5.1f} -> holdout median {med:7.1f} km | in-country {100*ok:5.1f}%")
    if best_med is None or med < best_med: best_med, best_w = med, w
    del m; gc.collect(); torch.cuda.empty_cache()
W_PROVIDED = best_w
log(f">>> chosen W_PROVIDED = {W_PROVIDED}")

[05:03:56 | + 151.9 min]   W_PROVIDED=  3.0 -> holdout median   647.3 km | in-country  48.5%
[05:05:43 | + 153.7 min]   W_PROVIDED=  8.0 -> holdout median   625.9 km | in-country  48.4%
[05:07:31 | + 155.5 min]   W_PROVIDED= 20.0 -> holdout median   628.5 km | in-country  48.1%
[05:07:31 | + 155.5 min] >>> chosen W_PROVIDED = 8.0


In [12]:
# =====================================================================
# CELL 12 - Seed ensemble on ALL trainable data, predicted on the holdout
# This is the key fix: the holdout predictions are produced by exactly the
# same ensemble average that the test set will see.
# =====================================================================
models = []
Hf = torch.zeros(len(HOLD), CFG.N_FINE, device=device)
Hc = torch.zeros(len(HOLD), N_COUNTRY, device=device)
Hd = torch.zeros(len(HOLD), 3, device=device)
Hu = torch.zeros(len(HOLD), device=device)

for s in range(CFG.N_MODELS):
    log(f"=== model {s+1}/{CFG.N_MODELS} ===")
    m = train_model(TRAINABLE, CFG.SEED+s, W_PROVIDED)
    pf,pc,pd_,pu = head_predict(m, HOLD)
    Hf+=pf; Hc+=pc; Hd+=pd_; Hu+=pu
    models.append(m)
    torch.save(m.state_dict(), os.path.join(CFG.ART, f"head_seed{s}.pt"))
Hf/=CFG.N_MODELS; Hc/=CFG.N_MODELS; Hd/=CFG.N_MODELS; Hu/=CFG.N_MODELS

v = decode_point(torch.log(Hf+1e-12), Hd)
la, lo = vec_to_latlon(v.cpu().numpy())
log(f"ENSEMBLE holdout median error: {np.median(hav_km(la,lo,hold_lat,hold_lon)):.1f} km")
log(f"ENSEMBLE holdout in-country  : "
    f"{100*((assign_country(la,lo)==hold_ctry)&(hold_ctry>=0)).mean():.1f}%")

[05:07:31 | + 155.5 min] === model 1/5 ===
[05:07:35 | + 155.6 min]     seed42 ep 1/60 | loss 15.3093
[05:08:39 | + 156.7 min]     seed42 ep 16/60 | loss 6.7526
[05:09:43 | + 157.7 min]     seed42 ep 31/60 | loss 5.9550
[05:10:47 | + 158.8 min]     seed42 ep 46/60 | loss 5.7581
[05:11:46 | + 159.8 min]     seed42 ep 60/60 | loss 5.7113
[05:11:46 | + 159.8 min] === model 2/5 ===
[05:11:50 | + 159.8 min]     seed43 ep 1/60 | loss 15.2666
[05:12:54 | + 160.9 min]     seed43 ep 16/60 | loss 6.7926
[05:13:57 | + 162.0 min]     seed43 ep 31/60 | loss 6.0005
[05:15:01 | + 163.0 min]     seed43 ep 46/60 | loss 5.7590
[05:16:01 | + 164.0 min]     seed43 ep 60/60 | loss 5.7129
[05:16:01 | + 164.0 min] === model 3/5 ===
[05:16:05 | + 164.1 min]     seed44 ep 1/60 | loss 15.2844
[05:17:09 | + 165.2 min]     seed44 ep 16/60 | loss 6.7514
[05:18:13 | + 166.2 min]     seed44 ep 31/60 | loss 6.0040
[05:19:17 | + 167.3 min]     seed44 ep 46/60 | loss 5.7595
[05:20:16 | + 168.3 min]     seed44 ep 60/60 

In [13]:
# =====================================================================
# CELL 13 - Optional extra member: last run's fine-tuned backbone.
# It was cut off mid-LR-schedule so it is undertrained, not bad - and it
# fails on different images than the frozen probe, which is what makes an
# ensemble worth having. Auto-disabled if it looks misaligned.
# =====================================================================
FT_OK = False; Ff=Fc=Fd=Fu=None
if CFG.USE_FT_MEMBER and PREV_FT:
    try:
        ck = torch.load(PREV_FT[0], map_location="cpu")
        ftb = CLIPVisionModel.from_pretrained(CLIP_LOCAL).to(device).half().eval()
        ftb.load_state_dict({k:v.half() for k,v in ck["backbone"].items()})
        fth = GeoHead(2048, CFG.N_FINE, CFG.N_COARSE, N_COUNTRY, CFG.HID).to(device)
        fth.load_state_dict(ck["head"]); fth.eval()
        MEANC = torch.tensor([0.48145466,0.4578275,0.40821073],device=device).view(1,3,1,1).half()
        STDC  = torch.tensor([0.26862954,0.26130258,0.27577711],device=device).view(1,3,1,1).half()

        @torch.no_grad()
        def ft_predict(paths, jitter=0.0):
            dl = DataLoader(ViewDS(paths,jitter), batch_size=64, shuffle=False,
                            num_workers=CFG.WORKERS, pin_memory=True)
            A,B_,C_,D_ = [],[],[],[]
            for x in dl:
                b = x.shape[0]
                x = x.to(device).reshape(b*N_VIEWS,3,CFG.IMG,CFG.IMG).half().div_(255.)
                f = ftb(pixel_values=(x-MEANC)/STDC).pooler_output.reshape(b,2048)
                fl,cl,ctl,dl_,ul = fth(f)
                A.append(torch.softmax(fl.float(),1)); B_.append(torch.softmax(ctl.float(),1))
                C_.append(dl_.float()); D_.append(ul.float())
            return torch.cat(A),torch.cat(B_),torch.cat(C_),torch.cat(D_)

        Ff,Fc,Fd,Fu = ft_predict(full_df.path.values[HOLD])
        vf = decode_point(torch.log(Ff+1e-12), Fd)
        lf, of_ = vec_to_latlon(vf.cpu().numpy())
        med_ft = float(np.median(hav_km(lf,of_,hold_lat,hold_lon)))
        log(f"fine-tuned member holdout median: {med_ft:.1f} km")
        FT_OK = med_ft < 2000.0
        log(f"fine-tuned member {'ACCEPTED' if FT_OK else 'REJECTED (misaligned cells?)'}")
    except Exception as e:
        log(f"fine-tuned member unavailable: {e}")
else:
    log("no fine-tuned member attached")

[05:29:03 | + 177.0 min] fine-tuned member unavailable: mat1 and mat2 must have the same dtype, but got Half and Float


In [14]:
# =====================================================================
# CELL 14 - Joint policy search on the holdout: (blend, lambda, alpha, floor)
# The metric is unpublished, so score the worst case over a family of
# plausible decay scales rather than betting on one.
# =====================================================================
D_GRID = [500.,1000.,1500.,2000.]
def proxy(d,r,ok,D,W=.4,CB=.15,RT=750.):
    return np.exp(-d/D) + W*np.exp(-r/D)*np.where(d<=r,1.,-1.) + CB*(ok&(r<=RT)).astype(float)
def maximin(d,r,ok): return min(np.median(proxy(d,r,ok,D)) for D in D_GRID)

BLENDS = [0.0,0.15,0.3,0.45] if FT_OK else [0.0]
LAMS   = [0.0,0.15,0.25,0.4,0.6,1.0]
best = None
for bw in BLENDS:
    pf = (1-bw)*Hf + (bw*Ff if FT_OK else 0)
    pc = (1-bw)*Hc + (bw*Fc if FT_OK else 0)
    pdl= (1-bw)*Hd + (bw*Fd if FT_OK else 0)
    pu = ((1-bw)*Hu + (bw*Fu if FT_OK else 0)).cpu().numpy()
    u  = np.expm1(pu)
    for lam in LAMS:
        vv = decode_point(country_adjust(torch.log(pf+1e-12), torch.log(pc+1e-12), lam), pdl)
        la2, lo2 = vec_to_latlon(vv.cpu().numpy())
        e2 = hav_km(la2, lo2, hold_lat, hold_lon)
        ok2 = (assign_country(la2,lo2)==hold_ctry)&(hold_ctry>=0)
        for a in np.arange(0.4, 8.01, 0.2):
            for fl_ in [15.,30.,60.,120.]:
                r2 = np.clip(a*np.maximum(u,1.0), fl_, CFG.R_MAX)
                s2 = maximin(e2, r2, ok2)
                if best is None or s2 > best[0]:
                    best = (s2, bw, lam, float(a), float(fl_),
                            float(np.median(e2)), float(ok2.mean()))
SCORE, BW, LAM, ALPHA, FLOOR, MED, OKR = best
log(f">>> blend={BW} lambda={LAM} alpha={ALPHA:.1f} floor={FLOOR:.0f}")
log(f"    holdout median {MED:.1f} km | in-country {100*OKR:.1f}% | score {SCORE:.4f}")

pf = (1-BW)*Hf + (BW*Ff if FT_OK else 0)
pu = ((1-BW)*Hu + (BW*Fu if FT_OK else 0)).cpu().numpy()
r_h = np.clip(ALPHA*np.maximum(np.expm1(pu),1.0), FLOOR, CFG.R_MAX)
log(f"    coverage {100*np.mean(hav_km(*vec_to_latlon(decode_point(country_adjust(torch.log(pf+1e-12), torch.log(((1-BW)*Hc+(BW*Fc if FT_OK else 0))+1e-12), LAM), (1-BW)*Hd+(BW*Fd if FT_OK else 0)).cpu().numpy()), hold_lat, hold_lon) <= r_h):.1f}% "
    f"| median radius {np.median(r_h):.0f} km")
json.dump({"blend":BW,"lam":LAM,"alpha":ALPHA,"floor":FLOOR,"score":SCORE,
           "w_provided":W_PROVIDED,"use_dino":bool(CFG.USE_DINO)},
          open(os.path.join(CFG.ART,"calibration.json"),"w"))
np.save(os.path.join(CFG.ART,"fine_centroids.npy"), CF)
np.save(os.path.join(CFG.ART,"cell_country.npy"), CELL_COUNTRY)

[05:29:07 | + 177.1 min] >>> blend=0.0 lambda=0.0 alpha=4.2 floor=15
[05:29:07 | + 177.1 min]     holdout median 583.1 km | in-country 50.4% | score 0.3772
[05:29:07 | + 177.1 min]     coverage 58.4% | median radius 801 km


In [15]:
# =====================================================================
# CELL 15 - Test inference + submission
# =====================================================================
def encode_test(jitter=0.0):
    fs = []
    bk = load_backbone("clip"); fs.append(encode_with(bk, TEST_PATHS, "test-clip", jitter))
    bk["model"].cpu(); del bk; gc.collect(); torch.cuda.empty_cache()
    if CFG.USE_DINO:
        bk = load_backbone("dino"); fs.append(encode_with(bk, TEST_PATHS, "test-dino", jitter))
        bk["model"].cpu(); del bk; gc.collect(); torch.cuda.empty_cache()
    Z = np.concatenate(fs,1) if len(fs)>1 else fs[0]
    return torch.tensor((Z.astype(np.float32)-MU)/SD, dtype=torch.float32, device=device)

Xt = encode_test(0.0)
Tf = torch.zeros(len(Xt), CFG.N_FINE, device=device); Tc = torch.zeros(len(Xt), N_COUNTRY, device=device)
Td = torch.zeros(len(Xt), 3, device=device); Tu = torch.zeros(len(Xt), device=device)
with torch.no_grad():
    for m in models:
        fl,cl,ctl,dl_,ul = m(Xt)
        Tf += torch.softmax(fl.float(),1); Tc += torch.softmax(ctl.float(),1)
        Td += dl_.float(); Tu += ul.float()
Tf/=len(models); Tc/=len(models); Td/=len(models); Tu/=len(models)

if FT_OK and BW > 0:
    Gf,Gc,Gd,Gu = ft_predict(TEST_PATHS)
    Tf = (1-BW)*Tf + BW*Gf; Tc = (1-BW)*Tc + BW*Gc
    Td = (1-BW)*Td + BW*Gd; Tu = (1-BW)*Tu + BW*Gu
    log(f"blended fine-tuned member at weight {BW}")

v = decode_point(country_adjust(torch.log(Tf+1e-12), torch.log(Tc+1e-12), LAM), Td)
p_lat, p_lon = vec_to_latlon(v.cpu().numpy())
p_rad = np.clip(ALPHA*np.maximum(np.expm1(Tu.cpu().numpy()),1.0), FLOOR, CFG.R_MAX)

ocean = np.where(assign_country(p_lat,p_lon) < 0)[0]
log(f"points outside every country: {len(ocean)}")
for j in ocean:
    try:
        pt = Point(float(p_lon[j]), float(p_lat[j]))
        gi = CTREE.nearest(pt); gi = int(gi if np.isscalar(gi) else np.asarray(gi).ravel()[0])
        q,_ = nearest_points(country_geoms[gi], pt); p_lat[j], p_lon[j] = q.y, q.x
    except Exception: pass

p_lat = np.clip(p_lat,-90,90); p_lon = ((p_lon+180)%360)-180
bad = ~np.isfinite(p_lat)|~np.isfinite(p_lon)|~np.isfinite(p_rad)
p_lat[bad], p_lon[bad], p_rad[bad] = 0.0, 0.0, 2000.0

sub = sub_template.copy()
sub[SUB_LAT], sub[SUB_LON], sub[SUB_RAD] = p_lat, p_lon, p_rad
sub = sub[list(sub_template.columns)]
assert len(sub)==len(sub_template) and sub.isna().sum().sum()==0
OUT = os.path.join(CFG.WORK,"submission.csv"); sub.to_csv(OUT, index=False)
log(f"SUBMISSION -> {OUT}")
print(sub.head(8).to_string())
log(f"radius median {sub[SUB_RAD].median():.0f} km (min {sub[SUB_RAD].min():.0f}, max {sub[SUB_RAD].max():.0f})")
log("NOTE: holdout median radius was "
    f"{np.median(r_h):.0f} km - these should now be in the same ballpark, "
    "unlike the previous run (905 vs 638).")

[05:29:44 | + 177.7 min] points outside every country: 87
[05:29:44 | + 177.7 min] SUBMISSION -> /kaggle/working/submission.csv
               image_id   pred_lat    pred_lon  pred_radius_km
0  34f65e00cc3df67d.jpg  14.963739  -16.546369      791.529236
1  14fabfc3e0d31fc7.jpg -11.074229   27.213648      663.946594
2  88a891ffc6beabac.jpg  -5.176022  145.311158      562.627258
3  f8bec9bfff55065d.jpg  38.076590  -87.397630      789.415955
4  d16ea14697a3c421.jpg   7.792557  100.253754      631.128540
5  3b2cef528f285b83.jpg  19.130330  -88.867563      632.611938
6  af38392885b9f454.jpg  50.797472    7.226884      436.248901
7  1574541bd6040c32.jpg -23.140279   29.272999      662.378479
[05:29:44 | + 177.7 min] radius median 654 km (min 293, max 2018)
[05:29:44 | + 177.7 min] NOTE: holdout median radius was 801 km - these should now be in the same ballpark, unlike the previous run (905 vs 638).


In [16]:
# =====================================================================
# CELL 16 - Radius-probe variants + offline proof + summary
# =====================================================================
# Identical coordinates, radius scaled. Submitting these tells you which way
# the real (unpublished) calibration term wants your radii to move - the
# coordinates are unchanged, so every point of difference comes from the
# calibration term alone.
for tag, mult in [("half",0.5), ("double",2.0)]:
    s2 = sub.copy(); s2[SUB_RAD] = np.clip(sub[SUB_RAD]*mult, 5, CFG.R_MAX)
    s2.to_csv(os.path.join(CFG.WORK, f"submission_radius_{tag}.csv"), index=False)
log("wrote submission_radius_half.csv and submission_radius_double.csv")

os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
try:
    _ = CLIPVisionModel.from_pretrained(CLIP_LOCAL); del _
    log("OFFLINE CHECK: CLIP loads from local path with hub disabled [PASS]")
    if CFG.USE_DINO:
        _ = AutoModel.from_pretrained(DINO_LOCAL); del _
        log("OFFLINE CHECK: DINOv2 loads locally                     [PASS]")
    h = GeoHead(FEAT_DIM, CFG.N_FINE, CFG.N_COARSE, N_COUNTRY, CFG.HID)
    h.load_state_dict(torch.load(os.path.join(CFG.ART,"head_seed0.pt"), map_location="cpu")); del h
    log("OFFLINE CHECK: head loads from artifacts                [PASS]")
    log(">>> RULE 4.3 SATISFIED <<<")
except Exception as e:
    log(f"OFFLINE CHECK FAILED: {e}"); traceback.print_exc()
finally:
    os.environ.pop("HF_HUB_OFFLINE",None); os.environ.pop("TRANSFORMERS_OFFLINE",None)

log("="*72)
log(f"holdout median error : {MED:.1f} km      (prev run OOF: 680.6)")
log(f"holdout in-country   : {100*OKR:.1f}%    (prev run: 51.8%)")
log(f"W_PROVIDED           : {W_PROVIDED}")
log(f"blend / lambda       : {BW} / {LAM}")
log(f"radius               : {ALPHA:.1f} x pred_err, floor {FLOOR:.0f}")
log(f"backbones            : clip{' + dinov2' if CFG.USE_DINO else ''}")
log("="*72)

[05:29:44 | + 177.7 min] wrote submission_radius_half.csv and submission_radius_double.csv
[05:29:44 | + 177.7 min] OFFLINE CHECK: CLIP loads from local path with hub disabled [PASS]
[05:29:45 | + 177.7 min] OFFLINE CHECK: DINOv2 loads locally                     [PASS]
[05:29:45 | + 177.7 min] OFFLINE CHECK: head loads from artifacts                [PASS]
[05:29:45 | + 177.7 min] >>> RULE 4.3 SATISFIED <<<
[05:29:45 | + 177.7 min] ========================================================================
[05:29:45 | + 177.7 min] holdout median error : 583.1 km      (prev run OOF: 680.6)
[05:29:45 | + 177.7 min] holdout in-country   : 50.4%    (prev run: 51.8%)
[05:29:45 | + 177.7 min] W_PROVIDED           : 8.0
[05:29:45 | + 177.7 min] blend / lambda       : 0.0 / 0.0
[05:29:45 | + 177.7 min] radius               : 4.2 x pred_err, floor 15
[05:29:45 | + 177.7 min] backbones            : clip + dinov2
[05:29:45 | + 177.7 min] ==============================================================